In [1]:
# Viterbi Algorithm for Nature Primer 
import math

# Helper function to compute log safely
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Define the states
states = ['E', '5', 'I']

# Define transition probabilities in log-space
transition = {
    'E': {'E': log(0.9), '5': log(0.1), 'I': log(0.0)},
    '5': {'I': log(1.0)},
    'I': {'I': log(0.9), 'E': log(0.1)}
}

# Define emission probabilities in log-space
emission = {
    'E': {'A': log(0.25), 'C': log(0.25), 'G': log(0.25), 'T': log(0.25)},
    '5': {'A': log(0.05), 'C': log(0.0),  'G': log(0.95), 'T': log(0.0)},
    'I': {'A': log(0.4), 'C': log(0.1), 'G': log(0.1), 'T': log(0.4)}
}


# 1. Function to compute log-prob of a known path & seq

def get_log_prob_of_a_given_path(path, sequence):
    """
    Computes the total log-probability of emitting a sequence
    from a given hidden path.
    """
    total_log_prob = 0.0
    for i in range(len(sequence)):
        state = path[i]
        base = sequence[i]
        emit = emission[state][base]
        total_log_prob += emit
        if i > 0:
            total_log_prob += transition[path[i-1]][state]
    return round(total_log_prob, 2)

# Example usage
path = "EEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
print("Log probability of given path:", get_log_prob_of_a_given_path(path, sequence[:len(path)]))

Log probability of given path: -35.83


In [2]:
# ---------------------------------------------------------
# 2. Full Viterbi Algorithm Implementation
# ---------------------------------------------------------
def viterbi(sequence, states, start_prob, transition, emission):
    """
    Implements the Viterbi algorithm to find the most probable
    path of hidden states given an observed sequence.
    """
    V = [{}]  # DP table
    path = {}

    # Initialization
    for state in states:
        V[0][state] = start_prob[state] + emission[state][sequence[0]]
        path[state] = [state]

    # Dynamic programming
    for t in range(1, len(sequence)):
        V.append({})
        new_path = {}

        for curr_state in states:
            max_prob, prev_state = max(
                ((V[t-1][prev_state] + transition[prev_state].get(curr_state, -math.inf), prev_state)
                 for prev_state in states),
                key=lambda x: x[0]
            )
            V[t][curr_state] = max_prob + emission[curr_state].get(sequence[t], -math.inf)
            new_path[curr_state] = path[prev_state] + [curr_state]

        path = new_path

    # Backtrace best final state
    max_final_state = max(V[-1], key=lambda s: V[-1][s])
    return path[max_final_state], V[-1][max_final_state]


In [3]:

# 3. Run Viterbi on example sequence

# Define uniform start probabilities
start_prob = {s: log(1.0 / len(states)) for s in states}

# Run Viterbi
viterbi_path, final_log_prob = viterbi(sequence, states, start_prob, transition, emission)

# Print results
print("Most likely path:")
print("".join(viterbi_path))
print("Final log-probability of most likely path:", round(final_log_prob, 2))


Most likely path:
EEEEEEEEEEEEEEEEEEEEEEEEEE
Final log-probability of most likely path: -39.78
